In [ ]:
# # Cyberbullying Detection System

# ## 1. Install Required Libraries

# %%
!pip install pandas scikit-learn numpy matplotlib seaborn --quiet


# ## 2. Upload Your Dataset


# %%
# from google.colab import files
import pandas as pd
import io

df = pd.read_csv("./cyberbullying_data_labelled.csv")

print(f"✅ Loaded! Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(df.head())

print("=" * 60)
print("✅ DATASET LOADED SUCCESSFULLY")
print("=" * 60)
print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\n📝 Columns found: {list(df.columns)}")


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


✅ Loaded! Rows: 239465
Columns: ['Text', 'oh_label', 'cleaned_text', 'tokens', 'processed_text', '__index_level_0__']
                                                Text  oh_label  \
0                             Get fucking real dude.         1   
1   She is as dirty as they come  and that crook ...         1   
2   why did you fuck it up. I could do it all day...         1   
3   Dude they dont finish enclosing the fucking s...         1   
4   WTF are you talking about Men? No men thats n...         1   

                                        cleaned_text  \
0                              get fucking real dude   
1  she is as dirty as they come and that crook re...   
2  why did you fuck it up i could do it all day t...   
3  dude they do not finish enclosing the fucking ...   
4  wtf are you talking about men no men that is n...   

                                              tokens  \
0                    ['get' 'fucking' 'real' 'dude']   
1  ['she' 'is' 'as' 'dirty' 'as' 'th

In [6]:
# 3. Explore Your Data

# Display first few rows
print("\n🔍 First 5 rows:")
print(df.head())

print("\n📈 Label Distribution:")
print(df['oh_label'].value_counts())
print(f"\nPercentages:\n{df['oh_label'].value_counts(normalize=True) * 100}")

# Check for missing values
print(f"\n⚠️ Missing values:\n{df.isnull().sum()}")



🔍 First 5 rows:
                                                Text  oh_label  \
0                             Get fucking real dude.         1   
1   She is as dirty as they come  and that crook ...         1   
2   why did you fuck it up. I could do it all day...         1   
3   Dude they dont finish enclosing the fucking s...         1   
4   WTF are you talking about Men? No men thats n...         1   

                                        cleaned_text  \
0                              get fucking real dude   
1  she is as dirty as they come and that crook re...   
2  why did you fuck it up i could do it all day t...   
3  dude they do not finish enclosing the fucking ...   
4  wtf are you talking about men no men that is n...   

                                              tokens  \
0                    ['get' 'fucking' 'real' 'dude']   
1  ['she' 'is' 'as' 'dirty' 'as' 'they' 'come' 'a...   
2  ['why' 'did' 'you' 'fuck' 'it' 'up' 'i' 'could...   
3  ['dude' 'they' 'do' 'n

In [8]:

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import numpy as np
import pickle

# Use your preprocessed text column if available, otherwise use Text
# Priority: processed_text > cleaned_text > Text
if 'processed_text' in df.columns:
    text_column = 'processed_text'
    print("✅ Using 'processed_text' column")
elif 'cleaned_text' in df.columns:
    text_column = 'cleaned_text'
    print("✅ Using 'cleaned_text' column")
else:
    text_column = 'Text'
    print("✅ Using 'Text' column")

# Remove any rows with missing values in text or labels
df_clean = df.dropna(subset=[text_column, 'oh_label'])

X = df_clean[text_column].astype(str)
y = df_clean['oh_label'].astype(int)

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📊 Training samples: {len(X_train)}")
print(f"📊 Testing samples: {len(X_test)}")



✅ Using 'processed_text' column

📊 Training samples: 191572
📊 Testing samples: 47893


In [9]:

# ## 5. Create TF-IDF Vectorizer and Train Model

# %%
# Create TF-IDF vectorizer (converts text to numbers)
vectorizer = TfidfVectorizer(
    max_features=5000,      # Top 5000 words
    ngram_range=(1, 2),     # Use single words and word pairs
    stop_words='english',   # Remove common words (the, is, at)
    min_df=2,               # Ignore words appearing in less than 2 documents
    max_df=0.8              # Ignore words appearing in more than 80% of documents
)

# Transform text to TF-IDF features
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"✅ TF-IDF matrix shape: {X_train_tfidf.shape}")

# Train Logistic Regression model (fast and effective)
model = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
model.fit(X_train_tfidf, y_train)


✅ TF-IDF matrix shape: (191572, 5000)


LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [11]:

# Evaluate model
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n🎯 Model Accuracy: {accuracy:.2%}")
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Bullying', 'Bullying']))

# %% [markdown]
# ## 6. Prediction Function (Paste Comment Here!)

# %%
def predict_bullying(comment_text):
    """
    Predict if a comment is cyberbullying
    
    Args:
        comment_text (str): The comment to analyze
        
    Returns:
        dict: Prediction results
    """
    # Preprocess input (same preprocessing as your dataset)
    processed_comment = str(comment_text).lower().strip()
    
    # Transform using same vectorizer
    comment_tfidf = vectorizer.transform([processed_comment])
    
    # Predict
    prediction = model.predict(comment_tfidf)[0]
    probabilities = model.predict_proba(comment_tfidf)[0]
    
    # Get confidence scores
    not_bullying_conf = probabilities[0] * 100
    bullying_conf = probabilities[1] * 100
    
    # Determine label and risk
    if prediction == 1:
        label = "🚨 BULLYING"
        risk_level = "HIGH RISK" if bullying_conf > 80 else "MODERATE RISK" if bullying_conf > 60 else "LOW RISK"
        recommendation = "⚠️ This comment should be flagged and reviewed."
    else:
        label = "✅ NOT BULLYING"
        risk_level = "SAFE"
        recommendation = "✓ This comment appears safe."
    
    return {
        'input_text': comment_text,
        'prediction': label,
        'confidence': f"{max(bullying_conf, not_bullying_conf):.1f}%",
        'bullying_probability': f"{bullying_conf:.1f}%",
        'safe_probability': f"{not_bullying_conf:.1f}%",
        'risk_level': risk_level,
        'recommendation': recommendation
    }

def display_prediction(comment):
    """Display formatted prediction results"""
    result = predict_bullying(comment)
    
    print("=" * 70)
    print("🛡️ CYBERBULLYING DETECTION RESULT")
    print("=" * 70)
    print(f"📝 Comment: {result['input_text']}")
    print(f"\n🔍 Result: {result['prediction']}")
    print(f"📊 Confidence: {result['confidence']}")
    print(f"⚖️ Risk Level: {result['risk_level']}")
    print(f"\n📈 Probabilities:")
    print(f"   🔴 Bullying: {result['bullying_probability']}")
    print(f"   🟢 Safe: {result['safe_probability']}")
    print(f"\n💡 {result['recommendation']}")
    print("=" * 70)



🎯 Model Accuracy: 86.21%

📋 Classification Report:
              precision    recall  f1-score   support

Not Bullying       0.97      0.87      0.92     42385
    Bullying       0.44      0.79      0.57      5508

    accuracy                           0.86     47893
   macro avg       0.71      0.83      0.74     47893
weighted avg       0.91      0.86      0.88     47893



In [14]:

# %% [markdown]
# ## 7. Test with Sample Comments

# %%
# Test examples
test_comments = [
    "You're so stupid and ugly, nobody likes you",
    "Go kill yourself loser",
    "I think you made a good point in class today",
    "Thanks for helping me with the homework",
    "Fat pig, stop eating so much",
    "You're worthless and should die",
    "Can we meet at the library tomorrow?",
    "Everyone hates you, just disappear"
]

print("🧪 TESTING WITH SAMPLE COMMENTS:\n")
for i, comment in enumerate(test_comments, 1):
    print(f"\nTest #{i}:")
    display_prediction(comment)

# %% [markdown]
# ## 8. Interactive Input (Paste Your Own Comment!)

# %%
print("\n" + "=" * 70)
print("🔍 INTERACTIVE MODE - Paste any comment to check")
print("=" * 70)
print("Type 'quit' to exit\n")

while True:
    user_input = input(">>> Enter comment: ")
    
    if user_input.lower() in ['quit', 'exit', 'q']:
        print("\n👋 Exiting... Stay safe online!")
        break
    
    if not user_input.strip():
        print("⚠️ Please enter a comment.\n")
        continue
    
    display_prediction(user_input)
    print("\n")


🧪 TESTING WITH SAMPLE COMMENTS:


Test #1:
🛡️ CYBERBULLYING DETECTION RESULT
📝 Comment: You're so stupid and ugly, nobody likes you

🔍 Result: 🚨 BULLYING
📊 Confidence: 99.9%
⚖️ Risk Level: HIGH RISK

📈 Probabilities:
   🔴 Bullying: 99.9%
   🟢 Safe: 0.1%

💡 ⚠️ This comment should be flagged and reviewed.

Test #2:
🛡️ CYBERBULLYING DETECTION RESULT
📝 Comment: Go kill yourself loser

🔍 Result: 🚨 BULLYING
📊 Confidence: 99.4%
⚖️ Risk Level: HIGH RISK

📈 Probabilities:
   🔴 Bullying: 99.4%
   🟢 Safe: 0.6%

💡 ⚠️ This comment should be flagged and reviewed.

Test #3:
🛡️ CYBERBULLYING DETECTION RESULT
📝 Comment: I think you made a good point in class today

🔍 Result: ✅ NOT BULLYING
📊 Confidence: 83.3%
⚖️ Risk Level: SAFE

📈 Probabilities:
   🔴 Bullying: 16.7%
   🟢 Safe: 83.3%

💡 ✓ This comment appears safe.

Test #4:
🛡️ CYBERBULLYING DETECTION RESULT
📝 Comment: Thanks for helping me with the homework

🔍 Result: ✅ NOT BULLYING
📊 Confidence: 91.6%
⚖️ Risk Level: SAFE

📈 Probabilities:
   🔴 Bullyi

🛡️ CYBERBULLYING DETECTION RESULT
📝 Comment: I love my mother

🔍 Result: 🚨 BULLYING
📊 Confidence: 65.6%
⚖️ Risk Level: MODERATE RISK

📈 Probabilities:
   🔴 Bullying: 65.6%
   🟢 Safe: 34.4%

💡 ⚠️ This comment should be flagged and reviewed.


🛡️ CYBERBULLYING DETECTION RESULT
📝 Comment: i love sleeping

🔍 Result: ✅ NOT BULLYING
📊 Confidence: 84.3%
⚖️ Risk Level: SAFE

📈 Probabilities:
   🔴 Bullying: 15.7%
   🟢 Safe: 84.3%

💡 ✓ This comment appears safe.


🛡️ CYBERBULLYING DETECTION RESULT
📝 Comment: I love eating food

🔍 Result: ✅ NOT BULLYING
📊 Confidence: 86.2%
⚖️ Risk Level: SAFE

📈 Probabilities:
   🔴 Bullying: 13.8%
   🟢 Safe: 86.2%

💡 ✓ This comment appears safe.


🛡️ CYBERBULLYING DETECTION RESULT
📝 Comment: I love sleeping

🔍 Result: ✅ NOT BULLYING
📊 Confidence: 84.3%
⚖️ Risk Level: SAFE

📈 Probabilities:
   🔴 Bullying: 15.7%
   🟢 Safe: 84.3%

💡 ✓ This comment appears safe.


🛡️ CYBERBULLYING DETECTION RESULT
📝 Comment: I love eating food

🔍 Result: ✅ NOT BULLYING
📊 Confidence: 

In [15]:

# ## 9. Save Model for Later Use


# Save the model and vectorizer
model_filename = 'cyberbullying_model.pkl'
vectorizer_filename = 'tfidf_vectorizer.pkl'

pickle.dump(model, open(model_filename, 'wb'))
pickle.dump(vectorizer, open(vectorizer_filename, 'wb'))

print(f"💾 Model saved as: {model_filename}")
print(f"💾 Vectorizer saved as: {vectorizer_filename}")


print("\n✅ Files downloaded! You can load them later using:")
print("   model = pickle.load(open('cyberbullying_model.pkl', 'rb'))")
print("   vectorizer = pickle.load(open('tfidf_vectorizer.pkl', 'rb'))")



💾 Model saved as: cyberbullying_model.pkl
💾 Vectorizer saved as: tfidf_vectorizer.pkl

✅ Files downloaded! You can load them later using:
   model = pickle.load(open('cyberbullying_model.pkl', 'rb'))
   vectorizer = pickle.load(open('tfidf_vectorizer.pkl', 'rb'))


In [ ]:
# ## 10. Load and Use Saved Model (Future Use)

# %%
"""
# To load the saved model later:
import pickle

# Load model and vectorizer
model = pickle.load(open('cyberbullying_model.pkl', 'rb'))
vectorizer = pickle.load(open('tfidf_vectorizer.pkl', 'rb'))

# Predict
comment = "Your comment here"
comment_tfidf = vectorizer.transform([comment])
prediction = model.predict(comment_tfidf)
probability = model.predict_proba(comment_tfidf)

print("Bullying" if prediction[0] == 1 else "Not Bullying")
print(f"Confidence: {max(probability[0])*100:.1f}%")
"""